# 📦 Dataset Collection & Publishing Pipeline

This notebook collects and enriches raw metadata from **IMDb**, **TMDB**, and **AniList**, then processes the data and publishes the final datasets to the **Hugging Face Hub** under the **UERP_Dataset** repository.

### 📤 Output Files
- `final_dataset_v1.parquet` — Combined IMDb + TMDB enriched dataset
- `anilist_final_v1.parquet` — Processed AniList dataset

> **Purpose:** Build a centralized, high-quality metadata repository for the Universal Entertainment Recommendation Platform (UERP).

In [1]:
import pandas as pd
import urllib.request
import os
from rich import print as rprint

os.makedirs('/kaggle/working/data', exist_ok = True)

urls = {
    'title.basics.tsv.gz': 'https://datasets.imdbws.com/title.basics.tsv.gz',
    'title.ratings.tsv.gz': 'https://datasets.imdbws.com/title.ratings.tsv.gz',
}

for filename, url in urls.items():
    filepath = f'/kaggle/working/data/{filename}'
    if not os.path.exists(filepath):
        rprint(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, filepath)
        rprint(f"Done: {filename}")
    else:
        rprint(f"{filename} already exists, skipping.")

Downloading title.basics.tsv.gz...

Done: title.basics.tsv.gz

Downloading title.ratings.tsv.gz...

Done: title.ratings.tsv.gz

In [2]:
basics = pd.read_csv(
    '/kaggle/working/data/title.basics.tsv.gz',
    sep = '\t',
    na_values = '\\N',
    low_memory = False
)

ratings = pd.read_csv(
    '/kaggle/working/data/title.ratings.tsv.gz',
    sep = '\t',
    na_values = '\\N'
)

rprint(f"Basics Shape: {basics.shape}")
rprint(f"Ratings Shape: {ratings.shape}")
display(basics.head())

Basics Shape: (12654261, 9)

Ratings Shape: (1696535, 3)

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894.0,NaN,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892.0,NaN,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892.0,NaN,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,Un bon bock,0,1892.0,NaN,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893.0,NaN,1,Short


In [3]:
rprint(basics['titleType'].value_counts())

titleType
tvEpisode       9779498
short           1145079
movie            752419
video            328356
tvSeries         302468
tvMovie          155447
tvMiniSeries      71652
tvSpecial         58675
videoGame         49639
tvShort           11027
tvPilot               1
Name: count, dtype: int64

In [4]:
merged = basics.merge(ratings, on = 'tconst', how = 'inner')
rprint(f"After merge: {merged.shape}")

relevant_types = ['movie', 'tvSeries', 'tvMiniSeries', 'tvMovie', 'tvShort', 'short', 'tvSpecial']
filtered = merged[merged['titleType'].isin(relevant_types).copy()]
rprint(f"After type filter: {filtered.shape}")
rprint(f"{filtered['titleType'].value_counts()}")

rprint(f"{filtered['numVotes'].describe()}")
rprint(f"{filtered['numVotes'].quantile([0.5, 0.7, 0.8, 0.9, 0.95, 0.99])}")

After merge: (1696535, 11)

After type filter: (746391, 11)

titleType
movie           348028
short           185560
tvSeries        113263
tvMovie          56986
tvMiniSeries     25699
tvSpecial        14252
tvShort           2603
Name: count, dtype: int64

count    7.463910e+05
mean     2.045038e+03
std      2.729878e+04
min      5.000000e+00
25%      1.400000e+01
50%      3.400000e+01
75%      1.510000e+02
max      3.208129e+06
Name: numVotes, dtype: float64

0.50       34.0
0.70      103.0
0.80      236.0
0.90      834.0
0.95     2579.0
0.99    30929.0
Name: numVotes, dtype: float64

In [5]:
threshold = 2579
subset = filtered[filtered['numVotes'] >= threshold].copy()

rprint(f"Total subset size: {subset.shape[0]}")
rprint()
rprint(f"Breakdown by titleType (after threshold)")
rprint(subset['titleType'].value_counts())
rprint()
rprint(f"Original vs Filtered % retained per type:")
for t in relevant_types:
    orig = (filtered['titleType'] == t).sum()
    kept = (subset['titleType'] == t).sum()
    pct = (kept / orig * 100) if orig > 0 else 0
    rprint(f"{t:15s} original = {orig:7d} kept = {kept:6d} retained = {pct:.1f}%")

Total subset size: 37331

Breakdown by titleType (after threshold)

titleType
movie           28219
tvSeries         5952
tvMiniSeries     1392
tvMovie           974
short             539
tvSpecial         221
tvShort            34
Name: count, dtype: int64

Original vs Filtered % retained per type:

movie           original =  348028 kept =  28219 retained = 8.1%

...

tvMiniSeries    original =   25699 kept =   1392 retained = 5.4%

tvMovie         original =   56986 kept =    974 retained = 1.7%

tvShort         original =    2603 kept =     34 retained = 1.3%

short           original =  185560 kept =    539 retained = 0.3%

tvSpecial       original =   14252 kept =    221 retained = 1.6%

In [8]:
target_counts = {
    'movie':        20000,
    'tvSeries':      8000,
    'tvMiniSeries':  2000,
    'tvMovie':       2000,
    'tvSpecial':     1000,
    'short':         1500,
    'tvShort':        500,
}

frames = []

for t, n in target_counts.items():
    grp = filtered[filtered['titleType'] == t].sort_values('numVotes', ascending = False)
    take = min(n, len(grp))
    frames.append(grp.head(take))

subset2 = pd.concat(frames, ignore_index = True)

rprint(f"Total subset size: {subset2.shape[0]}")
rprint()
rprint(subset2['titleType'].value_counts())
rprint()
rprint("Min numVotes per type (implied cutoff):")
rprint(subset2.groupby('titleType')['numVotes'].min())

Total subset size: 35000

titleType
movie           20000
tvSeries         8000
tvMiniSeries     2000
tvMovie          2000
short            1500
tvSpecial        1000
tvShort           500
Name: count, dtype: int64

Min numVotes per type (implied cutoff):

titleType
movie           4571
short           1251
tvMiniSeries    1549
tvMovie         1558
tvSeries        1569
tvShort           93
tvSpecial        493
Name: numVotes, dtype: int64

In [9]:
os.makedirs('/kaggle/working/processed', exist_ok = True)

subset2.to_parquet('/kaggle/working/processed/imdb_subset_v1.parquet', index = False)
subset2.to_csv('/kaggle/working/processed/imdb_subset_v1.csv', index = False)

rprint(f"Saved: {subset2.shape}")
rprint(subset2.columns.tolist())

Saved: (35000, 11)

[
    'tconst',
    'titleType',
    'primaryTitle',
    'originalTitle',
    'isAdult',
    'startYear',
    'endYear',
    'runtimeMinutes',
    'genres',
    'averageRating',
    'numVotes'
]

## **TMDB Enrichment**

In [13]:
import requests

url = f"https://api.themoviedb.org/3/find/tt0111161"

params = {
    "api_key": TMDB_API_KEY, 
    "external_source": "imdb_id"
}

resp = requests.get(url, params = params, timeout = 10)

rprint("Status code:", resp.status_code)
rprint("Response text:", resp.text)

Status code: 200

Response text: 
{"movie_results":[{"adult":false,"backdrop_path":"/zfbjgQE1uSd9wiPTX4VzsLi0rGG.jpg","id":278,"title":"The Shawshank
Redemption","original_title":"The Shawshank Redemption","overview":"Imprisoned in the 1940s for the double murder 
of his wife and her lover, upstanding banker Andy Dufresne begins a new life at the Shawshank prison, where he puts
his accounting skills to work for an amoral warden. During his long stretch in prison, Dufresne comes to be admired
by the other inmates -- including an older prisoner named Red -- for his integrity and unquenchable sense of 
hope.","poster_path":"/9cqNxx0GxF0bflZmeSMuL5tnGzr.jpg","media_type":"movie","original_language":"en","genre_ids":[
18,80],"popularity":62.3273,"release_date":"1994-09-23","softcore":false,"video":false,"vote_average":8.7,"vote_cou
nt":30790}],"person_results":[],"tv_results":[],"tv_episode_results":[],"tv_season_results":[]}

In [12]:
from kaggle_secrets import UserSecretsClient
import time
import requests

user_secrets = UserSecretsClient()

TMDB_API_KEY = user_secrets.get_secret("TMDB_API_KEY")

def fetch_tmdb_by_imdb_id(imdb_id, api_key):
    url = f"https://api.themoviedb.org/3/find/{imdb_id}"
    params = {
        "api_key": api_key,
        "external_source": "imdb_id"
    }
    resp = requests.get(url, params = params, timeout = 10)
    if resp.status_code != 200:
        return None

    return resp.json()

sample = subset2.head(5)

for _, row in sample.iterrows():
    result = fetch_tmdb_by_imdb_id(row['tconst'], TMDB_API_KEY)
    rprint(row['tconst'], row['primaryTitle'], '->')
    rprint(result)
    rprint('---')
    time.sleep(0.3)

tt0111161 The Shawshank Redemption ->

{
    'movie_results': [
        {
            'adult': False,
            'backdrop_path': '/zfbjgQE1uSd9wiPTX4VzsLi0rGG.jpg',
            'id': 278,
            'title': 'The Shawshank Redemption',
            'original_title': 'The Shawshank Redemption',
            'overview': 'Imprisoned in the 1940s for the double murder of his wife and her lover, upstanding banker
Andy Dufresne begins a new life at the Shawshank prison, where he puts his accounting skills to work for an amoral 
warden. During his long stretch in prison, Dufresne comes to be admired by the other inmates -- including an older 
prisoner named Red -- for his integrity and unquenchable sense of hope.',
            'poster_path': '/9cqNxx0GxF0bflZmeSMuL5tnGzr.jpg',
            'media_type': 'movie',
            'original_language': 'en',
            'genre_ids': [18, 80],
            'popularity': 62.3273,
            'release_date': '1994-09-23',
            'softcore': False,
            'video': False,
            'vote_average': 8.7,
            'vote_count': 30790
        }
    ],
    'person_results': [],
    'tv_results': [],
    'tv_episode_results': [],
    'tv_season_results': []
}

---

tt0468569 The Dark Knight ->

{
    'movie_results': [
        {
            'adult': False,
            'backdrop_path': '/dqK9Hag1054tghRQSqLSfrkvQnA.jpg',
            'id': 155,
            'title': 'The Dark Knight',
            'original_title': 'The Dark Knight',
            'overview': 'Batman raises the stakes in his war on crime. With the help of Lt. Jim Gordon and District
Attorney Harvey Dent, Batman sets out to dismantle the remaining criminal organizations that plague the streets. 
The partnership proves to be effective, but they soon find themselves prey to a reign of chaos unleashed by a 
rising criminal mastermind known to the terrified citizens of Gotham as the Joker.',
            'poster_path': '/qJ2tW6WMUDux911r6m7haRef0WH.jpg',
            'media_type': 'movie',
            'original_language': 'en',
            'genre_ids': [28, 80, 53],
            'popularity': 22.3965,
            'release_date': '2008-07-16',
            'softcore': False,
            'video': False,
            'vote_average': 8.533,
            'vote_count': 36122
        }
    ],
    'person_results': [],
    'tv_results': [],
    'tv_episode_results': [],
    'tv_season_results': []
}

...

{
    'movie_results': [
        {
            'adult': False,
            'backdrop_path': '/c6OLXfKAk5BKeR6broC8pYiCquX.jpg',
            'id': 550,
            'title': 'Fight Club',
            'original_title': 'Fight Club',
            'overview': 'A ticking-time-bomb insomniac and a slippery soap salesman channel primal male aggression 
into a shocking new form of therapy. Their concept catches on, with underground "fight clubs" forming in every 
town, until an eccentric gets in the way and ignites an out-of-control spiral toward oblivion.',
            'poster_path': '/jSziioSwPVrOy9Yow3XhWIBDjq1.jpg',
            'media_type': 'movie',
            'original_language': 'en',
            'genre_ids': [18, 53],
            'popularity': 16.8242,
            'release_date': '1999-10-15',
            'softcore': False,
            'video': False,
            'vote_average': 8.438,
            'vote_count': 32360
        }
    ],
    'person_results': [],
    'tv_results': [],
    'tv_episode_results': [],
    'tv_season_results': []
}

---

tt0816692 Interstellar ->

{
    'movie_results': [
        {
            'adult': False,
            'backdrop_path': '/2ssWTSVklAEc98frZUQhgtGHx7s.jpg',
            'id': 157336,
            'title': 'Interstellar',
            'original_title': 'Interstellar',
            'overview': 'The adventures of a group of explorers who make use of a newly discovered wormhole to 
surpass the limitations on human space travel and conquer the vast distances involved in an interstellar voyage.',
            'poster_path': '/yQvGrMoipbRoddT0ZR8tPoR7NfX.jpg',
            'media_type': 'movie',
            'original_language': 'en',
            'genre_ids': [12, 18, 878],
            'popularity': 57.4104,
            'release_date': '2014-11-05',
            'softcore': False,
            'video': False,
            'vote_average': 8.482,
            'vote_count': 40336
        }
    ],
    'person_results': [],
    'tv_results': [],
    'tv_episode_results': [],
    'tv_season_results': []
}

---

## **Pipeline Design for $35k$**

In [29]:
from concurrent.futures import ThreadPoolExecutor, as_completed

CHECKPOINT_PATH = '/kaggle/working/processed/tmdb_enriched.parquet'
FAILED_LOG_PATH = '/kaggle/working/processed/tmdb_failed.csv'
BATCH_SIZE = 500
MAX_WORKERS = 8

def fetch_tmdb_by_imdb_id(imdb_id, api_key):
    url = f"https://api.themoviedb.org/3/find/{imdb_id}"
    params = {
        "api_key": api_key,
        "external_source": "imdb_id"
    }

    try:
        resp = requests.get(url, params = params, timeout = 10)
        if resp.status_code != 200:
            return imdb_id, None, f"HTTP {resp.status_code}"

        data = resp.json()
        results = data.get('movie_results') or data.get('tv_results')

        if not results:
            return imdb_id, None, "no_match"

        r = results[0]

        return imdb_id, {
            'tmdb_id': r.get('id'),
            'overview': r.get('overview'),
            'poster_path': r.get('poster_path'),
            'backdrop_path': r.get('backdrop_path'),
            'vote_average': r.get('vote_average'),
            'popularity': r.get('popularity'),
            'genre_ids': r.get('genre_ids'),
        }, None
    except Exception as e:
        return imdb_id, None, str(0)

def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        df = pd.read_parquet(CHECKPOINT_PATH)
        return df, set(df['tconst'])
    return pd.DataFrame(), set()

def run_enrichment(target_df, api_key):
    enriched_df, done_ids = load_checkpoint()
    todo = target_df[~target_df['tconst'].isin(done_ids)]
    total = len(todo)
    rprint(f"Already done: {len(done_ids)} | Remaining: {total}")

    failed_records = []
    for start in range(0, total, BATCH_SIZE):
        batch = todo.iloc[start: start + BATCH_SIZE]
        batch_results = []
        with ThreadPoolExecutor(max_workers = MAX_WORKERS) as executor:
            futures = {
                executor.submit(fetch_tmdb_by_imdb_id, row['tconst'], api_key): row
                for _, row in batch.iterrows()
            }
            for future in as_completed(futures):
                row = futures[future]
                imdb_id, data, error = future.result()
                if data:
                    record = {
                        "tconst": imdb_id,
                        "primaryTitle": row['primaryTitle']
                    }
                    record.update(data)
                    batch_results.append(record)
                else:
                    failed_records.append(
                        {
                            "tconst": imdb_id,
                            'primaryTitle': row['primaryTitle'],
                            'error': error
                        }
                    )

        new_df = pd.DataFrame(batch_results)
        enriched_df = pd.concat([enriched_df, new_df], ignore_index = True) if not enriched_df.empty else new_df
        enriched_df.to_parquet(CHECKPOINT_PATH, index = False)

        if failed_records:
            pd.DataFrame(failed_records).to_csv(FAILED_LOG_PATH, index = False)

        rprint(f"Checkpoint saved: {min(start + BATCH_SIZE, total)}/{total} | Failed so far: {len(failed_records)}")
        time.sleep(0.5)

    rprint(f"DONE. Total enriched: {len(enriched_df)} | Total failed: {len(failed_records)}")
    return enriched_df

### **Testing on 300 titles**

In [31]:
test_result = run_enrichment(subset2.head(300), TMDB_API_KEY)
rprint(test_result.shape)
test_result.head()

Already done: 300 | Remaining: 0

DONE. Total enriched: 300 | Total failed: 0

(300, 9)

,tconst,primaryTitle,tmdb_id,overview,poster_path,backdrop_path,vote_average,popularity,genre_ids
0,tt0468569,The Dark Knight,155,Batman raises the stakes in his war on crime. ...,/qJ2tW6WMUDux911r6m7haRef0WH.jpg,/dqK9Hag1054tghRQSqLSfrkvQnA.jpg,8.533,22.3965,"[28, 80, 53]"
1,tt0110912,Pulp Fiction,680,"A burger-loving hit man, his philosophical par...",/vQWk5YBFWF4bZaofAbv0tShwBvQ.jpg,/suaEOtk1N1sgg2MTM7oZd2cfVp3.jpg,8.481,20.0809,"[53, 80, 35]"
2,tt0109830,Forrest Gump,13,A man with a low IQ has accomplished great thi...,/Cw4hIUIAmSYfK9QfaUW5igp9La.jpg,/66Kn4XWhkuPkJxOJyPEx4U2CUfN.jpg,8.465,15.7142,"[35, 18, 10749]"
3,tt1375666,Inception,27205,"Cobb, a skilled thief who commits corporate es...",/xlaY2zyzMfkhk0HSC5VUwzoZPU1.jpg,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,8.373,28.9174,"[28, 878, 12]"
4,tt0111161,The Shawshank Redemption,278,Imprisoned in the 1940s for the double murder ...,/9cqNxx0GxF0bflZmeSMuL5tnGzr.jpg,/zfbjgQE1uSd9wiPTX4VzsLi0rGG.jpg,8.700,62.3273,"[18, 80]"


In [32]:
full_result = run_enrichment(subset2, TMDB_API_KEY)
rprint(full_result.shape)

Already done: 300 | Remaining: 34700

Checkpoint saved: 500/34700 | Failed so far: 0

Checkpoint saved: 1000/34700 | Failed so far: 0

Checkpoint saved: 1500/34700 | Failed so far: 0

Checkpoint saved: 2000/34700 | Failed so far: 0

...

Checkpoint saved: 34000/34700 | Failed so far: 430

Checkpoint saved: 34500/34700 | Failed so far: 501

Checkpoint saved: 34700/34700 | Failed so far: 559

DONE. Total enriched: 34441 | Total failed: 559

(34441, 9)

In [35]:
failed_log = pd.read_csv('/kaggle/working/processed/tmdb_failed.csv')
failed_with_type = failed_log.merge(subset2[['tconst', 'titleType']], on = 'tconst', how = 'left')

rprint("Failed breakdown by titleType:")
rprint(failed_with_type['titleType'].value_counts())
rprint()

for t, target_n in target_counts.items():
    failed_n = (failed_with_type['titleType'] == t).sum()
    success_n = target_n - failed_n
    pct = success_n / target_n * 100
    rprint(f"{t:15s}  target = {target_n:6d}  enriched = {success_n:6d}  ({pct:.1f}%)")

Failed breakdown by titleType:

titleType
tvSeries        149
tvSpecial       119
tvShort         107
short            98
tvMiniSeries     44
tvMovie          32
movie            10
Name: count, dtype: int64

movie            target =  20000  enriched =  19990  (100.0%)

tvSeries         target =   8000  enriched =   7851  (98.1%)

tvMiniSeries     target =   2000  enriched =   1956  (97.8%)

tvMovie          target =   2000  enriched =   1968  (98.4%)

tvSpecial        target =   1000  enriched =    881  (88.1%)

short            target =   1500  enriched =   1402  (93.5%)

tvShort          target =    500  enriched =    393  (78.6%)

In [40]:
backfill_needed = {
    'tvSpecial': target_counts['tvSpecial'] - 881,   # 119 short
    'short':     target_counts['short'] - 1402,      # 98 short
    'tvShort':   target_counts['tvShort'] - 393,      # 107 short
}

rprint(backfill_needed)

backfill_candidates = []
for t, missing_n in backfill_needed.items():
    grp = filtered[filtered['titleType'] == t].sort_values('numVotes', ascending = False)
    target_n = target_counts[t]
    buffer_n = missing_n * 5 
    pool = grp.iloc[target_n : target_n + buffer_n]
    backfill_candidates.append(pool)

backfill_df = pd.concat(backfill_candidates, ignore_index = True)
rprint("Backfill pool size:", backfill_df.shape[0])
rprint(backfill_df['titleType'].value_counts())

{'tvSpecial': 119, 'short': 98, 'tvShort': 107}

Backfill pool size: 1620

titleType
tvSpecial    595
tvShort      535
short        490
Name: count, dtype: int64

In [41]:
backfill_result = run_enrichment(backfill_df, TMDB_API_KEY)
print(backfill_result.shape)

Already done: 35704 | Remaining: 357

Checkpoint saved: 357/357 | Failed so far: 357

DONE. Total enriched: 35704 | Total failed: 357

(35704, 9)


In [42]:
all_candidates = pd.concat([subset2, backfill_df], ignore_index=True).drop_duplicates('tconst')

enriched_all = pd.read_parquet(CHECKPOINT_PATH)
merged_final = enriched_all.merge(
    all_candidates[['tconst', 'titleType', 'numVotes', 'startYear', 'genres', 'runtimeMinutes', 'averageRating']],
    on = 'tconst', how = 'left'
)

final_frames = []
for t, n in target_counts.items():
    grp = merged_final[merged_final['titleType'] == t].sort_values('numVotes', ascending = False)
    final_frames.append(grp.head(n))

final_dataset = pd.concat(final_frames, ignore_index = True)
rprint("Final dataset size:", final_dataset.shape)
rprint(final_dataset['titleType'].value_counts())

Final dataset size:
(34765, 15)

titleType
movie           19990
tvSeries         7851
tvMovie          1968
tvMiniSeries     1956
short            1500
tvSpecial        1000
tvShort           500
Name: count, dtype: int64

In [44]:
final_dataset.to_parquet('/kaggle/working/processed/final_dataset_v1.parquet', index = False)
rprint("Saved locally:", final_dataset.shape)

Saved locally:
(34765, 15)

In [47]:
from huggingface_hub import HfApi, login

hf_secret = user_secrets.get_secret("HF_TOKEN") 
login(token = hf_secret)

api = HfApi()

api.upload_file(
    path_or_fileobj = '/kaggle/working/processed/final_dataset_v1.parquet',
    path_in_repo = 'final_dataset_v1.parquet',
    repo_id = 'Subhadip007/UERP_Dataset',
    repo_type = 'dataset',
)

rprint("Pushed to HF Hub!")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed to HF Hub!

## **JIKAN Anime**

In [1]:
import pandas as pd
import os
import time
import requests
from rich import print as rprint

from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

TMDB_API_KEY = user_secrets.get_secret("TMDB_API_KEY")
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

from huggingface_hub import login

login(token = HF_TOKEN)

os.makedirs('/kaggle/working/processed', exist_ok = True)

In [2]:
# JIKAN_BASE = "https://api.jikan.moe/v4"
# CHECKPOINT_PATH_ANIME = '/kaggle/working/processed/jikan_raw.parquet'
# PAGES_PER_CHECKPOINT = 20

In [2]:
# def fetch_anime_page_minimal(page, order_by = 'members', sort = 'desc', max_retries = 4):
#     url = f"{JIKAN_BASE}/anime"
#     params = {
#         "page": page,
#         "limit": 25,
#         # "order_by": order_by,
#         # "sort": sort,
#         # "sfw": "true"   # safe-for-work filter, for excluding adult contents
#     } 

#     for attempt in range(max_retries):
#         resp = requests.get(url, params = params, headers={"User-Agent": "Mozilla/5.0"}, timeout = 15)

#         # rprint(resp.url)
#         # rprint(resp.status_code)
#         # rprint(resp.text[:300])
        
#         if resp.status_code == 200:
#             return resp.json()
#         elif resp.status_code in (429, 500, 502, 503, 504):
#             wait = 2 ** attempt
#             rprint(f"Page {page}: got {resp.status_code}, retrying in {wait}s (attempt {attempt + 1}/{max_retries})")
#             time.sleep(wait)
#         else:
#             rprint(f"Page {page}: non-retryable error {resp.status_code} - {resp.text[:200]}")
#             return None

#     rprint(f"Page {page}: failed after {max_retries} retries")
#     return None


# def load_anime_checkpoint():
#     if os.path.exists(CHECKPOINT_PATH_ANIME):
#         df = pd.read_parquet(CHECKPOINT_PATH_ANIME)
#         return df, set(df['_source_page'])
#     return pd.DataFrame(), set()


# def run_anime_pull(total_pages, failed_pages_out = None):
#     all_df, done_pages = load_anime_checkpoint()
#     rprint(f"Already done pages: {len(done_pages)}")
#     buffer = []
#     failed_pages = []

#     for page in range(1, total_pages + 1):
#         if page in done_pages:
#             continue
#         data = fetch_anime_page_minimal(page)
#         if data is None:
#             failed_pages.append(page)
#             continue
#         for item in data.get('data', []):
#             buffer.append(
#                 {
#                     'mal_id': item.get('mal_id'),
#                     'title': item.get('title'),
#                     'type': item.get('type'),
#                     'episodes': item.get('episodes'),
#                     'score': item.get('score'),
#                     'scored_by': item.get('scored_by'),
#                     'members': item.get('members'),
#                     'popularity': item.get('popularity'),
#                     'year': item.get('year'),
#                     'rating': item.get('rating'),          # for local sfw filtering later
#                     'genres': [g['name'] for g in item.get('genres', [])],
#                     'synopsis': item.get('synopsis'),
#                     'image_url': item.get('images', {}).get('jpg', {}).get('image_url'),
#                     '_source_page': page,
#                 }
#             )
#         time.sleep(1.2)

#         if page % PAGES_PER_CHECKPOINT == 0:
#             new_df = pd.DataFrame(buffer)
#             all_df = pd.concat([all_df, new_df], ignore_index = True) if not all_df.empty else new_df
#             all_df.to_parquet(CHECKPOINT_PATH_ANIME, index = False)
#             buffer = []
#             rprint(f"Checkpoint saved: page {page}/{total_pages} | rows so far: {len(all_df)}")

#     if buffer:
#         new_df = pd.DataFrame(buffer)
#         all_df = pd.concat([all_df, new_df], ignore_index = True) if not all_df.empty else new_df
#         all_df.to_parquet(CHECKPOINT_PATH_ANIME, index = False)
#         rprint(f"Final checkpoint | total rows: {len(all_df)}")

#     if failed_pages_out is not None:
#         failed_pages_out.extend(failed_pages)

#     rprint(f"Pages failed this run: {failed_pages}")
#     return all_df

# rprint("Setup Complete. Ready to continue anime pull")

In [3]:
# failed = []

# test_anime = run_anime_pull(total_pages = 20, failed_pages_out = failed)

# rprint(test_anime.shape)
# rprint(test_anime['type'].value_counts())
# rprint(test_anime['rating'].value_counts())
# rprint(f"Failed Pages: {failed}")

In [4]:
ANILIST_URL = "https://graphql.anilist.co"

ANILIST_QUERY = """
query ($page: Int, $perPage: Int) {
  Page(page: $page, perPage: $perPage) {
    pageInfo { total currentPage lastPage hasNextPage }
    media(type: ANIME, sort: POPULARITY_DESC) {
      id
      idMal
      title { romaji english }
      format
      episodes
      averageScore
      popularity
      favourites
      seasonYear
      genres
      description
      coverImage { large }
      isAdult
    }
  }
}
"""

In [7]:
def fetch_anilist_page(page, per_page = 50, max_retries = 4):
    for attempt in range(max_retries):
        resp = requests.post(
            ANILIST_URL,
            json = {
                "query": ANILIST_QUERY,
                "variables": {
                    "page": page,
                    "perPage": per_page
                }
            },
            timeout = 15,
        )

        if resp.status_code == 200:
            return resp.json()
        elif resp.status_code == 429:
            retry_after = int(resp.headers.get("Retry-After", 60))
            rprint(f"Page {page}: rate limited, waiting {retry_after}s")
            time.sleep(retry_after)
        elif resp.status_code in (500, 502, 503, 504):
            wait = 2 ** attempt
            rprint(f"Page {page}: got {resp.status_code}, retrying in {wait}s")
            time.sleep(wait)
        else:
            rprint(f"Page {page}: non-retryable {resp.status_code} - {resp.text[:200]}")

    rprint(f"Page {page}: failed after {max_retries} retries")
    return None

In [8]:
test_records = []

for page in range(1, 4):
    data = fetch_anilist_page(page)
    if data is None:
        continue

    media_list = data.get('data', {}).get('Page', {}).get('media', [])

    for item in media_list:
        test_records.append(
            {
                'anilist_id': item.get('id'),
                'mal_id': item.get('idMal'),
                'title': item.get('title', {}).get('english') or item.get('title', {}).get('romaji'),
                'format': item.get('format'),
                'episodes': item.get('episodes'),
                'averageScore': item.get('averageScore'),
                'popularity': item.get('popularity'),
                'favourites': item.get('favourites'),
                'seasonYear': item.get('seasonYear'),
                'genres': item.get('genres'),
                'description': item.get('description'),
                'image_url': item.get('coverImage', {}).get('large'),
                'isAdult': item.get('isAdult'),
            }
        )

    time.sleep(0.8)     # ~75 req/min, safely under 90 limit

test_anilist_df = pd.DataFrame(test_records)
rprint(test_anilist_df.shape)
rprint(test_anilist_df['format'].value_counts())
rprint(test_anilist_df['isAdult'].value_counts())
rprint(test_anilist_df[['popularity', 'favourites', 'averageScore']].describe())
display(test_anilist_df.head())

(150, 13)

format
TV          138
MOVIE         9
ONA           2
TV_SHORT      1
Name: count, dtype: int64

isAdult
False    150
Name: count, dtype: int64

popularity     favourites  averageScore
count  1.500000e+02     150.000000    150.000000
mean   4.583133e+05   22335.753333     80.166667
std    1.537456e+05   16765.501759      5.868145
min    2.902600e+05    3948.000000     52.000000
25%    3.436868e+05   12097.000000     77.000000
50%    4.240015e+05   17447.500000     81.000000
75%    5.320362e+05   26244.500000     84.000000
max    1.030730e+06  107477.000000     91.000000

,anilist_id,mal_id,title,format,episodes,averageScore,popularity,favourites,seasonYear,genres,description,image_url,isAdult
0,16498,16498,Attack on Titan,TV,25.0,85,1030730,83865,2013,"[Action, Drama, Fantasy, Mystery]","Several hundred years ago, humans were nearly ...",https://s4.anilist.co/file/anilistcdn/media/an...,False
1,101922,38000,Demon Slayer: Kimetsu no Yaiba,TV,26.0,83,973809,59248,2019,"[Action, Adventure, Drama, Fantasy, Supernatural]","It is the Taisho Period in Japan. Tanjiro, a k...",https://s4.anilist.co/file/anilistcdn/media/an...,False
2,113415,40748,JUJUTSU KAISEN,TV,24.0,84,948353,68386,2020,"[Action, Drama, Supernatural]","A boy fights... for ""the right death.""<br>\n<b...",https://s4.anilist.co/file/anilistcdn/media/an...,False
3,1535,1535,Death Note,TV,37.0,84,939585,64567,2006,"[Mystery, Psychological, Supernatural, Thriller]",Light Yagami is a genius high school student w...,https://s4.anilist.co/file/anilistcdn/media/an...,False
4,21459,31964,My Hero Academia,TV,13.0,77,857297,34003,2016,"[Action, Adventure, Comedy]",What would the world be like if 80 percent of ...,https://s4.anilist.co/file/anilistcdn/media/an...,False


In [13]:
import re

def clean_html(text):
    if not text:
        return text
    text = re.sub('<[^<]+?>', '', text)   # To remove tags
    text = re.sub(r'\s+', ' ', text).strip()  # To remove extra whitespace

    return text

CHECKPOINT_PATH_ANILIST = '/kaggle/working/processed/anilist_raw.parquet'
PAGES_PER_CHECKPOINT = 20

def load_anilist_checkpoint():
    if os.path.exists(CHECKPOINT_PATH_ANILIST):
        df = pd.read_parquet(CHECKPOINT_PATH_ANILIST)
        return df, set(df['_source_page'])
    return pd.DataFrame(), set()


def run_anilist_pull():
    all_df, done_pages = load_anilist_checkpoint()
    rprint(f"Already done pages: {len(done_pages)}")
    buffer = []
    page = 1
    total_pages = None

    while True:
        if total_pages is not None and page > total_pages:
            break
        if page in done_pages:
            page += 1
            continue

        data = fetch_anilist_page(page)
        if data is None:
            page += 1
            continue

        page_info = data.get('data', {}).get('Page', {}).get('pageInfo', {})
        if total_pages is None:
            total_pages = page_info.get('lastPage', 400)
            rprint(f"Total pages detected: {total_pages}")

        media_list = data.get('data', {}).get('Page', {}).get('media', [])        
        for item in media_list:
            buffer.append(
                {
                    'anilist_id': item.get('id'),
                    'mal_id': item.get('idMal'),
                    'title': item.get('title', {}).get('english') or item.get('title', {}).get('romaji'),
                    'format': item.get('format'),
                    'episodes': item.get('episodes'),
                    'averageScore': item.get('averageScore'),
                    'popularity': item.get('popularity'),
                    'favourites': item.get('favourites'),
                    'seasonYear': item.get('seasonYear'),
                    'genres': item.get('genres'),
                    'description': clean_html(item.get('description')),
                    'image_url': item.get('coverImage', {}).get('large'),
                    'isAdult': item.get('isAdult'),
                    '_source_page': page,
                }
            )
        time.sleep(0.8)

        if page % PAGES_PER_CHECKPOINT == 0:
            new_df = pd.DataFrame(buffer)
            all_df = pd.concat([all_df, new_df], ignore_index = True) if not all_df.empty else new_df
            all_df.to_parquet(CHECKPOINT_PATH_ANILIST, index = False)
            buffer = []
            rprint(f"Checkpoint saved: page {page}/{total_pages} | rows so far: {len(all_df)}")

        page += 1
        if not page_info.get('hasNextPage', True):
            break

    if buffer:
        new_df = pd.DataFrame(buffer)
        all_df = pd.concat([all_df, new_df], ignore_index = True) if not all_df.empty else new_df
        all_df.to_parquet(CHECKPOINT_PATH_ANILIST, index = False)
        rprint(f"Final Checkpoint | total rows: {len(all_df)}")

    return all_df

In [14]:
anilist_full = run_anilist_pull()
rprint(anilist_full.shape)
rprint(anilist_full['format'].value_counts())
rprint(anilist_full['isAdult'].value_counts())

Already done pages: 0

Total pages detected: 100

Checkpoint saved: page 20/100 | rows so far: 1000

Page 31: rate limited, waiting 23s

Checkpoint saved: page 40/100 | rows so far: 2000

...

Page 92: rate limited, waiting 24s

Checkpoint saved: page 100/100 | rows so far: 5000

(5000, 14)

format
TV          2896
MOVIE        674
OVA          543
ONA          396
SPECIAL      304
TV_SHORT     164
MUSIC         12
Name: count, dtype: int64

isAdult
False    4924
True       76
Name: count, dtype: int64

In [17]:
resp = requests.post(
    ANILIST_URL,
    json={"query": ANILIST_QUERY, "variables": {"page": 1, "perPage": 50}},
    timeout=15,
)
page_info = resp.json()['data']['Page']['pageInfo']
rprint(page_info)

{'total': 5000, 'currentPage': 1, 'lastPage': 100, 'hasNextPage': True}

In [18]:
resp2 = requests.post(
    ANILIST_URL,
    json={"query": ANILIST_QUERY, "variables": {"page": 101, "perPage": 50}},
    timeout=15,
)
rprint(resp2.status_code)
rprint(resp2.json())

400

{
    'errors': [
        {
            'message': 'Page depth exceeds maximum allowed for API requests (5000 entries)',
            'status': 400,
            'locations': [{'line': 3, 'column': 3}]
        }
    ],
    'data': {'Page': None}
}

In [19]:
anilist_final = anilist_full[anilist_full['isAdult'] == False].copy()

rprint(f"Before Filter: {anilist_full.shape}")
rprint(f"After Filter: {anilist_final.shape}")
rprint(f"{anilist_final['format'].value_counts()}")

Before Filter: (5000, 14)

After Filter: (4924, 14)

format
TV          2890
MOVIE        671
OVA          496
ONA          384
SPECIAL      303
TV_SHORT     157
MUSIC         12
Name: count, dtype: int64

In [21]:
anilist_final2 = anilist_final[anilist_final['format'] != 'MUSIC'].copy()

rprint("Final anime count:", anilist_final2.shape[0])
rprint(anilist_final2['format'].value_counts())

anilist_final2.to_parquet('/kaggle/working/processed/anilist_final_v1.parquet', index = False)

Final anime count: 4912

format
TV          2890
MOVIE        671
OVA          496
ONA          384
SPECIAL      303
TV_SHORT     157
Name: count, dtype: int64

In [22]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_file(
    path_or_fileobj = '/kaggle/working/processed/anilist_final_v1.parquet',
    path_in_repo = 'anilist_final_v1.parquet',
    repo_id = 'Subhadip007/UERP_Dataset',
    repo_type = 'dataset',
)

rprint("Anilist data pushed to HF Hub!")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Anilist data pushed to HF Hub!